# 01. Exploração do Corpus e Fundamentos Linguísticos de NLP

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Fundamentação Teórica:** Fase 4 | Aula 01 - *Fundamentos de NLP e Representação de Linguagem*  

Antes de construir qualquer modelo de Machine Learning ou pipeline de IA Generativa (RAG), o cientista de dados precisa **diagnosticar e compreender o corpus textual**. A linguagem humana não é apenas texto estático: ela opera em múltiplas camadas de abstração linguística (morfologia, sintaxe, semântica e pragmática) e exibe regularidades estatísticas fundamentais como a **Lei de Zipf**.

### Objetivos deste Notebook:
1. Ingestão e diagnóstico inicial dos dados de avaliações do Olist (`olist_order_reviews_dataset`).
2. Análise estatística de corpus: total de tokens, vocabulário único e Type-Token Ratio (TTR).
3. Verificação empírica da **Lei de Zipf** em um corpus real em português brasileiro.
4. Investigação dos **Quatro Níveis Linguísticos** nos relatos dos clientes (Morfologia, Sintaxe, Semântica e Pragmática).
5. Mapeamento das particularidades do PT-BR informal e casos críticos de ambiguidade e ironia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuração estética dos gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

DATA_DIR = '../data/'

## 1. Carregamento e Auditoria dos Dados Brutos
Vamos carregar a base de avaliações do Olist e verificar a proporção de comentários textuais disponíveis.

In [ ]:
df_reviews_raw = pd.read_csv(f"{DATA_DIR}olist_order_reviews_dataset.csv")
total_registros = len(df_reviews_raw)

print(f"Total de avaliações registradas na base: {total_registros:,}")
print("\n--- Resumo de Tipos e Nulos ---")
print(df_reviews_raw.info())

nulos_mensagem = df_reviews_raw['review_comment_message'].isnull().sum()
pct_nulos = (nulos_mensagem / total_registros) * 100
print(f"\nAvaliações sem comentário de texto: {nulos_mensagem:,} ({pct_nulos:.2f}%)")

### Diagnóstico sobre os Dados Ausentes (Fase 4 | Aula 01):
- Mais de **58% dos clientes** deixam apenas a nota numérica (1 a 5 estrelas) sem redigir nenhuma palavra.
- Para um sistema de **Voice of Customer (VoC)** com RAG, vetores dependem do sinal semântico do texto. Indexar notas sem comentários no banco vetorial apenas poluiria o espaço latente com ruído.
- Portanto, nosso corpus de trabalho para NLP será composto estritamente pelos registros com texto preenchido.

In [ ]:
# Isolando o corpus com mensagens textuais
df_corpus = df_reviews_raw.dropna(subset=['review_comment_message']).copy()
df_corpus['review_comment_message'] = df_corpus['review_comment_message'].astype(str)

print(f"Tamanho final do corpus textual: {len(df_corpus):,} avaliações.")
print("\nDistribuição das notas neste corpus com texto:")
print(df_corpus['review_score'].value_counts(normalize=True).mul(100).round(2))

## 2. Estatísticas Básicas do Corpus (Tokens, Vocabulário e Riqueza Lexical)
Conforme introduzido na Aula 01, quantificamos as três grandezas fundamentais:
- **Tokens ($N$):** número total de ocorrências de palavras.
- **Vocabulário ($V$):** conjunto de termos únicos distintos.
- **Type-Token Ratio (TTR = $|V| / N$):** métrica clássica de riqueza/diversidade lexical.

In [ ]:
# Tokenização preliminar para análise estatística inicial do vocabulário bruto
def tokenizar_simples(texto):
    return re.findall(r'\b[a-záàâãäéèêëíìîïóòôõöúùûüç]+\b', texto.lower())

todos_tokens = []
tamanhos_comentarios = []

for texto in df_corpus['review_comment_message']:
    tokens = tokenizar_simples(texto)
    todos_tokens.extend(tokens)
    tamanhos_comentarios.append(len(tokens))

vocabulario_unico = set(todos_tokens)
total_tokens = len(todos_tokens)
tamanho_vocab = len(vocabulario_unico)
ttr = tamanho_vocab / total_tokens

print(f"Total de Tokens no Corpus (N): {total_tokens:,}")
print(f"Tamanho do Vocabulário Único (|V|): {tamanho_vocab:,}")
print(f"Type-Token Ratio (TTR): {ttr:.4f} ({ttr*100:.2f}%)")
print(f"Média de palavras por avaliação: {np.mean(tamanhos_comentarios):.1f} (mediana: {np.median(tamanhos_comentarios):.0f})")

In [ ]:
# Distribuição do comprimento das avaliações em palavras
plt.figure(figsize=(10, 4))
sns.histplot(tamanhos_comentarios, bins=50, kde=True, color="#2b5c8f")
plt.title("Distribuição do Comprimento das Avaliações (em Tokens)", fontsize=13, fontweight="bold")
plt.xlabel("Quantidade de Palavras")
plt.ylabel("Frequência")
plt.xlim(0, 70)
plt.show()

## 3. Verificação Empírica da Lei de Zipf (Aula 01)
A **Lei de Zipf** estabelece que a frequência de uma palavra é inversamente proporcional ao seu posto (rank) na tabela de frequências:  
$$f(r) \propto \frac{1}{r}$$

Em escala logarítmica, essa relação se traduz em uma reta com inclinação aproximada de $-1$. Vamos verificar se as avaliações reais de e-commerce brasileiro obedecem a essa lei da linguística quantitativa.

In [ ]:
contador_frequencias = Counter(todos_tokens)
top_20 = contador_frequencias.most_common(20)

df_top20 = pd.DataFrame(top_20, columns=['Palavra', 'Frequência'])
df_top20['Rank'] = range(1, len(df_top20) + 1)
print("Top 10 palavras mais frequentes no corpus:")
display(df_top20.head(10))

# Plot da Lei de Zipf em escala log-log
todas_freqs = [count for _, count in contador_frequencias.most_common()]
ranks = np.arange(1, len(todas_freqs) + 1)

plt.figure(figsize=(10, 5))
plt.loglog(ranks, todas_freqs, marker=".", linestyle="none", alpha=0.5, color="#d95f02", label="Frequência Observada")
# Linha teórica de Zipf (referência)
zipf_teorico = todas_freqs[0] / ranks
plt.loglog(ranks, zipf_teorico, linestyle="--", color="#1b9e77", label="Linha Teórica de Zipf (slope = -1)")

plt.title("Verificação da Lei de Zipf no Corpus do Olist (Escala Log-Log)", fontsize=13, fontweight="bold")
plt.xlabel("Rank da Palavra (log)")
plt.ylabel("Frequência Absoluta (log)")
plt.legend()
plt.show()

**Conclusão da Análise de Zipf:**  
O corpus do Olist replica com extrema fidelidade a Lei de Zipf: um número minúsculo de palavras gramaticais ("o", "de", "que", "não", "produto", "e") concentra a maior fatia das menções, enquanto a maior parte do vocabulário é formada por *hapax legomena* (palavras que aparecem apenas uma ou duas vezes).  

Destaque crítico: a palavra **"não"** é a 4ª palavra mais frequente de todo o corpus, comprovando que **negações** são elementos nucleares de relatos de compra.

## 4. Investigação dos Quatro Níveis Linguísticos no E-commerce (Aula 01)

Conforme a Aula 01, linguistas dividem o estudo da linguagem em 4 níveis que orientam diagnósticos de erro em sistemas de NLP:

| Nível | Objeto de Estudo | Impacto no Projeto Olist |
| :--- | :--- | :--- |
| **1. Morfologia** | Formação e flexão das palavras | PT-BR possui mais de 50 formas por verbo. Variações como "entregou", "entregar", "entregaram" precisam ser mapeadas. |
| **2. Sintaxe** | Combinação de palavras em frases | Pontuações expressivas ("!!!", "???") e elipses mudam o tom da avaliação. |
| **3. Semântica** | Significado isolado e polissemia | Palavras como "entrega", "prazo", "linha" assumem diferentes papéis contextuais. |
| **4. Pragmática** | Significado no contexto social / Ironia | Casos de sarcasmo como *"Adorei esperar 40 dias pelo meu produto quebrado"* desafiam baselines ingênuos. |

In [ ]:
# Investigando Casos de Pragmática e Ironia (Nota 1 com palavras positivas)
reviews_nota1 = df_corpus[df_corpus['review_score'] == 1]
ironias_candidatas = reviews_nota1[
    reviews_nota1['review_comment_message'].str.contains(r'\b(adorei|maravilh|parabéns|ótimo|otimo|perfeito)\b', case=False, regex=True)
]

print(f"Exemplos de avaliações Nota 1 com vocabulário aparentemente 'positivo' (Desafio de Pragmática):")
for i, texto in enumerate(ironias_candidatas['review_comment_message'].head(5), 1):
    print(f"[{i}] {texto}")

## 5. Síntese dos Fundamentos para o Próximo Passo

Com este diagnóstico completo do corpus:
1. Confirmamos que **limpar semântica sem critério** destrói a essência do dado (remover "não" inverteria completamente as queixas dos clientes).
2. Identificamos que contrações informais de redes sociais ("vc", "pq", "tbm", "obg", "n") são amplamente utilizadas pelos consumidores brasileiros.
3. Na **Parte 2 (Aula 02)**, implementaremos nosso **Pipeline Modular de Pré-processamento** para tratar esses pontos com idempotência e robustez.